In [4]:
import folium
import sqlalchemy 
import pandas as pd
import os
import sqlite3
from folium import FeatureGroup
from folium.plugins import HeatMap, MarkerCluster
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy import create_engine, Column, Integer, String, Enum, Float
from folium import IFrame
from branca.colormap import linear

# Combine two SQL databases together according to rank

In [5]:
# Define the directory for the databases
data_directory = "../Data/"

# Create engines for both databases with the directory paths
engine_main = create_engine(f"sqlite:///{data_directory}main.db", echo=False)
engine_places = create_engine(f"sqlite:///{data_directory}places.db", echo=False)

# Function to merge pokemon tables based on rank with different table names
def merge_pokemon_tables(main_table_name, places_table_name, engine_main, engine_places):
    # Load the tables from both databases into Pandas
    df_main = pd.read_sql(f"SELECT * FROM {main_table_name}", engine_main)
    df_places = pd.read_sql(f"SELECT * FROM {places_table_name}", engine_places)
    
    # Determine the smaller table size
    min_rows = min(len(df_main), len(df_places))
    
    # Sort by 'rank' and keep only the min_rows (to match the smaller table)
    df_main = df_main.sort_values(by="ranking").head(min_rows)
    df_places = df_places.sort_values(by="ranking").head(min_rows)
    
    # Perform a SQL-style JOIN on 'rank'
    merged_df = pd.merge(df_main, df_places, on="ranking", suffixes=("_main", "_places"))
    
    return merged_df

# Merge the ice_pokemon tables (with different names in main.db and places.db)
merged_ice_pokemon = merge_pokemon_tables("ice_pokemon", "coldest_places", engine_main, engine_places)

# Merge the fire_pokemon tables (with different names in main.db and places.db)
merged_fire_pokemon = merge_pokemon_tables("fire_pokemon", "hottest_places", engine_main, engine_places)

# Save merged tables into a NEW database inside the same directory
engine_merged = create_engine(f"sqlite:///{data_directory}merged.db", echo=False)

# Save the merged ice_pokemon and fire_pokemon tables to the merged database
merged_ice_pokemon.to_sql("ice_pokemon", engine_merged, if_exists="replace", index=False)
merged_fire_pokemon.to_sql("fire_pokemon", engine_merged, if_exists="replace", index=False)

print("✅ Merged tables saved to Data/merged.db")

✅ Merged tables saved to Data/merged.db


# Creating a map

In [ ]:
# # Define database path (adjust path if necessary)
# DATABASE_URL = "sqlite:///../Data/merged.db"

# # Connect to the database
# engine = create_engine(DATABASE_URL)
# Session = sessionmaker(bind=engine)
# session = Session()

# # Define ORM base
# Base = declarative_base()

# # Define ORM model for Fire Pokémon table
# class FirePokemon(Base):
#     __tablename__ = "fire_pokemon"
#     pokemon_id = Column(Integer, primary_key=True)
#     name = Column(String)
#     latitude = Column(Float)
#     longitude = Column(Float)
#     pokemon_description = Column(String)
#     pokemon_portrait = Column(String)
#     total_stat = Column(Float)
#     ranking = Column(Float)

# # Define ORM model for Ice Pokémon table
# class IcePokemon(Base):
#     __tablename__ = "ice_pokemon"
#     pokemon_id = Column(Integer, primary_key=True)
#     name = Column(String)
#     latitude = Column(Float)
#     longitude = Column(Float)
#     pokemon_description = Column(String)
#     pokemon_portrait = Column(String)
#     total_stat = Column(Float)
#     ranking = Column(Float)

# # Fetch Pokémon data from both tables
# fire_pokemon_entries = session.query(FirePokemon).all()
# ice_pokemon_entries = session.query(IcePokemon).all()

# # Initialize map
# map_center = [20.0, 0.0]
# pokemon_map = folium.Map(location=map_center, zoom_start=2)

# # Create feature groups for Fire and Ice Pokémon types
# fire_group = FeatureGroup(name="Fire Pokémon")
# ice_group = FeatureGroup(name="Ice Pokémon")

# # Function to add Pokémon markers with tooltips
# def add_pokemon_markers(pokemon_entries, group):
#     for pokemon in pokemon_entries:
#         lat, lon = pokemon.latitude, pokemon.longitude  # Use latitude and longitude directly

#         # Ensure values exist and use defaults if necessary
#         name = pokemon.name if pokemon.name else "Unknown"
#         description = pokemon.pokemon_description if pokemon.pokemon_description else "No description available"
#         stats = pokemon.total_stat if pokemon.total_stat else "N/A"
#         ranking = pokemon.ranking if pokemon.ranking else "Unranked"

#         # Tooltip content as plain text
#         tooltip_text = f"""
#         Name: {name}<br>
#         Stats: {stats}<br>
#         Ranking: {ranking}<br>
#         Description: {description}
#         """

#         # Create the tooltip with the plain text content
#         tooltip = folium.Tooltip(tooltip_text, sticky=True)  # 'sticky' makes it stay visible until you move the cursor

#         # Use the Pokémon image as a custom icon for the marker
#         custom_icon = folium.CustomIcon(pokemon.pokemon_portrait, icon_size=(80, 80))

#         # Add the marker to the group with the tooltip
#         marker = folium.Marker(location=[lat, lon], icon=custom_icon)
#         marker.add_child(tooltip)  # Attach the tooltip to the marker
#         group.add_child(marker)

# # Add Fire Pokémon markers
# add_pokemon_markers(fire_pokemon_entries, fire_group)

# # Add Ice Pokémon markers
# add_pokemon_markers(ice_pokemon_entries, ice_group)

# # Add groups to the map
# pokemon_map.add_child(fire_group)
# pokemon_map.add_child(ice_group)

In [6]:
# Define database path
DATABASE_URL = "sqlite:///../Data/merged.db"

# Connect to the database
engine = create_engine(DATABASE_URL)
Session = sessionmaker(bind=engine)
session = Session()

# Define ORM base
Base = declarative_base()

# Define ORM model for Fire Pokémon
class FirePokemon(Base):
    __tablename__ = "fire_pokemon"
    pokemon_id = Column(Integer, primary_key=True)
    name = Column(String)
    latitude = Column(Float)
    longitude = Column(Float)
    pokemon_description = Column(String)
    pokemon_portrait = Column(String)
    total_stat = Column(Float)
    ranking = Column(Float)

# Define ORM model for Ice Pokémon
class IcePokemon(Base):
    __tablename__ = "ice_pokemon"
    pokemon_id = Column(Integer, primary_key=True)
    name = Column(String)
    latitude = Column(Float)
    longitude = Column(Float)
    pokemon_description = Column(String)
    pokemon_portrait = Column(String)
    total_stat = Column(Float)
    ranking = Column(Float)

# Fetch Pokémon data
fire_pokemon_entries = session.query(FirePokemon).all()
ice_pokemon_entries = session.query(IcePokemon).all()

# Initialize the map
map_center = [20.0, 0.0]  # World center
pokemon_map = folium.Map(location=map_center, zoom_start=2)

# Create MarkerCluster groups for Fire and Ice Pokémon
fire_cluster = MarkerCluster(name="Fire Pokémon").add_to(pokemon_map)
ice_cluster = MarkerCluster(name="Ice Pokémon").add_to(pokemon_map)

def add_pokemon_markers(pokemon_entries, cluster_group):
    for pokemon in pokemon_entries:
        lat, lon = pokemon.latitude, pokemon.longitude  

        # Ensure values exist
        name = pokemon.name if pokemon.name else "Unknown"
        description = pokemon.pokemon_description if pokemon.pokemon_description else "No description available"
        stats = pokemon.total_stat if pokemon.total_stat else "N/A"
        ranking = pokemon.ranking if pokemon.ranking else "Unranked"
        portrait_url = pokemon.pokemon_portrait if pokemon.pokemon_portrait else "https://via.placeholder.com/150"
    
        
        tooltip_html = f"""
        <div style="text-align: center; width: 220px; max-width: 220px; padding: 10px; background-color: white; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); word-wrap: break-word; overflow-wrap: break-word;">
            <h4 style="font-size: 16px; font-weight: bold; color: #D34B29; margin: 0;">{name}</h4>
            <img src="{portrait_url}" width="150px" style="cursor: pointer; border-radius: 8px; max-width: 100%; height: auto;">
            <br><br>
            <b style="font-size: 14px;">Stats:</b> <span style="font-size: 12px; color: #555;">{stats}</span><br>
            <b style="font-size: 14px;">Ranking:</b> <span style="font-size: 12px; color: #555;">{ranking}</span><br>
            <p style="font-size: 12px; text-align: justify; color: #333; margin-top: 8px; line-height: 1.5; word-wrap: break-word; overflow-wrap: break-word; white-space: normal; max-width: 180px; padding: 0; margin: 0;">{description}</p>
        </div>
        """


        tooltip = folium.Tooltip(tooltip_html, sticky=True)  # sticky=True keeps the tooltip visible on hover

        # Create a DivIcon with the Pokémon image
        icon_html = f"""
        <div style="
            background: url('{portrait_url}') no-repeat center center;
            background-size: contain;
            width: 50px; height: 50px;">
        </div>
        """
        div_icon = folium.DivIcon(html=icon_html)

        # Create a marker with the Pokémon image and tooltip
        marker = folium.Marker(
            location=[lat, lon],
            tooltip=tooltip,  # Use Tooltip instead of Popup
            icon=div_icon  # Use Pokémon image as the marker icon
        )

        # Add marker to the cluster group
        marker.add_to(cluster_group)

# Add Fire Pokémon markers
add_pokemon_markers(fire_pokemon_entries, fire_cluster)

# Add Ice Pokémon markers
add_pokemon_markers(ice_pokemon_entries, ice_cluster)

# Add groups to the map
pokemon_map.add_child(fire_cluster)
pokemon_map.add_child(ice_cluster)

/tmp/ipykernel_17838/3938787984.py:10: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


# Getting hottest and coldest places


In [7]:
filename_cold = '/files/ds105a-2024-project-error_105/Temperature exploration/coldest_places.csv'
filename_hot = '/files/ds105a-2024-project-error_105/Temperature exploration/hottest_places.csv'

# Read CSV files
df_coldest_places = pd.read_csv(filename_cold)
df_hottest_places = pd.read_csv(filename_hot)

hotcold_marker_toggle = 0

if hotcold_marker_toggle == 1:
    # Create color scales for hot and cold places
    hot_colormap = linear.Reds_09.scale(df_hottest_places['temperature'].min(), df_hottest_places['temperature'].max())
    cold_colormap = linear.Blues_09.scale(df_coldest_places['temperature'].min(), df_coldest_places['temperature'].max())

    # Create feature groups for toggleable layers
    hot_layer = folium.FeatureGroup(name="Hottest Places")
    cold_layer = folium.FeatureGroup(name="Coldest Places")

    # Add markers for hottest places
    for idx, row in df_hottest_places.iterrows():
        rank = idx + 1
        folium.Marker(
            location=[row['latitude'], row['longitude']],
            popup=f"Rank: {rank}<br>Region: {row['region']}<br>Temperature: {row['temperature']}°C",
            icon=folium.Icon(color='red', icon='cloud'),
            tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)",
        ).add_to(hot_layer)

        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=10,
            color=hot_colormap(row['temperature']),
            fill=True,
            fill_opacity=0.8,
            tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)"
        ).add_to(hot_layer)

    # Add markers for coldest places
    for idx, row in df_coldest_places.iterrows():
        rank = idx + 1
        folium.Marker(
            location=[row['latitude'], row['longitude']],
            popup=f"Rank: {rank}<br>Region: {row['region']}<br>Temperature: {row['temperature']}°C",
            icon=folium.Icon(color='blue', icon='cloud'),
            tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)",
        ).add_to(cold_layer)

        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=10,
            color=cold_colormap(row['temperature']),
            fill=True,
            fill_opacity=0.8,
            tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)"
        ).add_to(cold_layer)

    # Add feature groups to the map
    pokemon_map.add_child(hot_layer)
    pokemon_map.add_child(cold_layer)

    # Add legends
    hot_colormap.caption = 'Temperature Scale (Hottest Places)'
    cold_colormap.caption = 'Temperature Scale (Coldest Places)'
    hot_colormap.add_to(pokemon_map)
    cold_colormap.add_to(pokemon_map)

# HeatMap for hottest and coldest places


In [8]:
# Create a color scale for temperatures
colormap = linear.RdYlBu_11.scale(df_coldest_places['temperature'].min(), df_hottest_places['temperature'].max())

# Function to map temperature to color intensity
def get_marker_color(temp, temp_min, temp_max, color_scale):
    # Map temperature to a hex color
    hex_color = color_scale(temp)
    return hex_color

heatmap_layer = folium.FeatureGroup(name="HeatMap")

# Add markers for hottest places
for _, row in df_hottest_places.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        popup=f"Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        color=get_marker_color(row['temperature'], df_coldest_places['temperature'].min(), 
                               df_hottest_places['temperature'].max(), colormap),
        fill=True,
        fill_color=get_marker_color(row['temperature'], df_coldest_places['temperature'].min(), 
                                    df_hottest_places['temperature'].max(), colormap),
        fill_opacity=0.8
    ).add_to(heatmap_layer)

# Add markers for coldest places
for _, row in df_coldest_places.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        popup=f"Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        color=get_marker_color(row['temperature'], df_coldest_places['temperature'].min(), 
                               df_hottest_places['temperature'].max(), colormap),
        fill=True,
        fill_color=get_marker_color(row['temperature'], df_coldest_places['temperature'].min(), 
                                    df_hottest_places['temperature'].max(), colormap),
        fill_opacity=0.8
    ).add_to(heatmap_layer)

# Add color scale legend to map
colormap.caption = 'Temperature Scale (°C)'
colormap.add_to(pokemon_map)

# Add HeatMap for temperature intensity
heat_data = [[row['latitude'], row['longitude'], row['temperature']] 
             for _, row in pd.concat([df_hottest_places, df_coldest_places]).iterrows()]
HeatMap(heat_data).add_to(heatmap_layer)


pokemon_map.add_child(heatmap_layer)

# Saving File


In [9]:
folium.LayerControl().add_to(pokemon_map)

# Save the map to an HTML file
pokemon_map.save("pokemon_map.html")

print("✅ Map generated: pokemon_map.html")


session.close()

✅ Map generated: pokemon_map.html
